# COLMAP BA v2 — pycolmap-cuda12 with Frame/Rig API

**Runtime**: GPU (T4) — Runtime > Change runtime type

**Input**: Upload `scan_data.zip` from Desktop

In [ ]:
# Cell 1: Install
!pip install pycolmap-cuda12
import pycolmap
print("version:", pycolmap.__version__, "| has_cuda:", pycolmap.has_cuda)

In [ ]:
# Cell 2: Upload scan_data.zip
from google.colab import files
import os, zipfile
uploaded = files.upload()
os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile('scan_data.zip', 'r') as z:
    z.extractall('/content/work')
print('Images:', len(os.listdir('/content/work/images')))
print('Sparse:', os.listdir('/content/work/sparse'))

In [ ]:
# Cell 3: Feature Extraction (GPU)
import pycolmap, os
os.chdir('/content/work')
if os.path.exists('database.db'):
    os.remove('database.db')
reader_opts = pycolmap.ImageReaderOptions()
reader_opts.camera_model = 'PINHOLE'
reader_opts.camera_params = '1428.3756,1428.3756,958.9948,725.38055'
pycolmap.extract_features(database_path='database.db', image_path='images', camera_mode=pycolmap.CameraMode.SINGLE, reader_options=reader_opts, device=pycolmap.Device.cuda)
print('=== Feature extraction complete ===')

In [ ]:
# Cell 4: Exhaustive Matching (GPU)
import pycolmap, os
os.chdir('/content/work')
opts = pycolmap.FeatureMatchingOptions()
opts.use_gpu = True
pycolmap.match_exhaustive(database_path='database.db', matching_options=opts)
print('=== Matching complete ===')

In [ ]:
# Cell 5: Create Reconstruction with ARKit poses (Frame/Rig API)
import sqlite3, numpy as np, os, shutil, pycolmap
os.chdir('/content/work')

arkit_poses = {}
with open('sparse/images.txt') as f:
    for line in f:
        if line.startswith('#') or line.strip() == '':
            continue
        parts = line.strip().split()
        if len(parts) >= 10 and parts[9].endswith('.jpg'):
            arkit_poses[parts[9]] = [float(x) for x in parts[1:8]]

conn = sqlite3.connect('database.db')
cursor = conn.execute('SELECT image_id, name FROM images')
db_images = [(img_id, name) for img_id, name in cursor.fetchall() if name in arkit_poses]
conn.close()
print(f'DB images matched to ARKit poses: {len(db_images)}')

rec = pycolmap.Reconstruction()
cam = pycolmap.Camera(model='PINHOLE', width=1920, height=1440, params=[1428.3756, 1428.3756, 958.9948, 725.38055], camera_id=1)
rec.add_camera(cam)

rig = pycolmap.Rig()
rig.rig_id = 1
sensor = pycolmap.sensor_t(type=pycolmap.SensorType.CAMERA, id=cam.camera_id)
rig.add_ref_sensor(sensor)
rec.add_rig(rig)

count = 0
for img_id, name in db_images:
    qw, qx, qy, qz, tx, ty, tz = arkit_poses[name]
    image = pycolmap.Image(image_id=img_id, name=name, camera_id=cam.camera_id)
    frame = pycolmap.Frame()
    frame.frame_id = img_id
    frame.rig_id = rig.rig_id
    frame.add_data_id(image.data_id)
    rec.add_frame(frame)
    rigid = pycolmap.Rigid3d(rotation=pycolmap.Rotation3d(np.array([qw, qx, qy, qz])), translation=np.array([tx, ty, tz]))
    rec.frame(frame.frame_id).set_cam_from_world(cam.camera_id, rigid)
    rec.add_image(image)
    count += 1

if os.path.exists('sparse_arkit_bin'):
    shutil.rmtree('sparse_arkit_bin')
os.makedirs('sparse_arkit_bin')
rec.write(output_path='sparse_arkit_bin')
print(f'Registered {count} images')
print(rec.summary())

In [ ]:
# Cell 6: Triangulate + Bundle Adjustment
import pycolmap, os, shutil
os.chdir('/content/work')

rec = pycolmap.Reconstruction('sparse_arkit_bin')
print(f'Before triangulation: {rec.summary()}')

if os.path.exists('sparse_triangulated'):
    shutil.rmtree('sparse_triangulated')
os.makedirs('sparse_triangulated')

rec = pycolmap.triangulate_points(reconstruction=rec, database_path='database.db', image_path='images', output_path='sparse_triangulated', refine_intrinsics=False)
print(f'After triangulation: {rec.summary()}')

ba_options = pycolmap.BundleAdjustmentOptions()
ba_options.refine_focal_length = False
ba_options.refine_principal_point = False
ba_options.refine_extra_params = False
pycolmap.bundle_adjustment(rec, ba_options)
print(f'After BA: {rec.summary()}')

if os.path.exists('sparse_refined_txt'):
    shutil.rmtree('sparse_refined_txt')
os.makedirs('sparse_refined_txt')
rec.write_text('sparse_refined_txt')
print(f'Saved to sparse_refined_txt/')
print('Files:', os.listdir('sparse_refined_txt'))

In [ ]:
# Cell 7: Compare pose corrections
import numpy as np

def parse_poses(path):
    poses = {}
    with open(path) as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.strip().split()
            if len(parts) >= 10 and parts[9].endswith('.jpg'):
                poses[parts[9]] = [float(x) for x in parts[1:8]]
    return poses

def cam_pos(qw,qx,qy,qz,tx,ty,tz):
    R = np.array([
        [1-2*(qy*qy+qz*qz), 2*(qx*qy-qw*qz), 2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz), 1-2*(qx*qx+qz*qz), 2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy), 2*(qy*qz+qw*qx), 1-2*(qx*qx+qy*qy)]
    ])
    return -R.T @ np.array([tx,ty,tz])

orig = parse_poses('/content/work/sparse/images.txt')
refined = parse_poses('/content/work/sparse_refined_txt/images.txt')
common = sorted(set(orig) & set(refined))

diffs = np.array([np.linalg.norm(cam_pos(*orig[n]) - cam_pos(*refined[n])) for n in common])
print(f'Common: {len(common)} | Mean: {diffs.mean()*100:.1f}cm | Max: {diffs.max()*100:.1f}cm')
print('Expected: 1-3cm corrections = BA working correctly')

In [ ]:
# Cell 8: Download refined poses
import zipfile, os
from google.colab import files
os.chdir('/content/work')
with zipfile.ZipFile('/content/refined_output.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir('sparse_refined_txt'):
        zf.write(f'sparse_refined_txt/{f}', f'sparse_refined/{f}')
files.download('/content/refined_output.zip')